# SQL Analysis - Pink Slip Data

In [ ]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    host='localhost',
    dbname='pinks',
    user='postgres',
    password=os.environ.get('DB_PASSWORD')
)

print('Connected to PostgreSQL')
pd.read_sql_query('SELECT COUNT(*) AS total_slips FROM pink_slip', conn)

## Revenue by Month

In [ ]:
pd.read_sql_query('''
    SELECT
        TO_CHAR(date_received, 'YYYY-MM') AS month,
        COUNT(*) AS total_slips,
        ROUND(SUM(total_amount), 2) AS revenue,
        ROUND(AVG(total_amount), 2) AS avg_slip_value
    FROM pink_slip
    GROUP BY month
    ORDER BY month
''', conn)

## Top 10 Customers by Total Spend

In [ ]:
pd.read_sql_query('''
    SELECT
        first_initial || '. ' || last_name AS customer,
        phone,
        COUNT(*) AS visits,
        ROUND(SUM(total_amount)::numeric, 2) AS total_spent,
        ROUND(AVG(total_amount)::numeric, 2) AS avg_per_visit
    FROM pink_slip
    GROUP BY first_initial, last_name, phone
    ORDER BY total_spent DESC
    LIMIT 10
''', conn)

## Item Type Breakdown

Revenue and volume by item type.

In [ ]:
pd.read_sql_query('''
    SELECT
        i.item_type,
        COUNT(*) AS total_items,
        ROUND(AVG(i.price)::numeric, 2) AS avg_price,
        ROUND(SUM(i.price)::numeric, 2) AS total_revenue,
        ROUND((SUM(i.price) * 100.0 / (SELECT SUM(price) FROM pink_slip_item))::numeric, 1) AS pct_of_revenue
    FROM pink_slip_item i
    INNER JOIN pink_slip s ON i.slip_id = s.id
    GROUP BY i.item_type
    ORDER BY total_revenue DESC
''', conn)

## Most Common Alterations by Item Type

In [ ]:
pd.read_sql_query('''
    SELECT
        item_type,
        work_description,
        COUNT(*) AS times_performed,
        ROUND(AVG(price)::numeric, 2) AS avg_price
    FROM pink_slip_item
    WHERE work_description IS NOT NULL AND work_description != ''
    GROUP BY item_type, work_description
    HAVING COUNT(*) > 10
    ORDER BY item_type, times_performed DESC
''', conn)

## Customer Retention Analysis

Segmenting customers by visit frequency to see which groups drive the most revenue.

In [ ]:
pd.read_sql_query('''
    WITH customer_visits AS (
        SELECT
            phone,
            first_initial || '. ' || last_name AS customer,
            COUNT(*) AS visit_count,
            ROUND(SUM(total_amount)::numeric, 2) AS lifetime_value
        FROM pink_slip
        GROUP BY phone, first_initial, last_name
    )
    SELECT
        CASE
            WHEN visit_count = 1 THEN 'One-time'
            WHEN visit_count BETWEEN 2 AND 4 THEN 'Repeat (2-4)'
            WHEN visit_count BETWEEN 5 AND 9 THEN 'Regular (5-9)'
            ELSE 'VIP (10+)'
        END AS customer_segment,
        COUNT(*) AS num_customers,
        ROUND((COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customer_visits))::numeric, 1) AS pct_of_customers,
        ROUND(SUM(lifetime_value)::numeric, 2) AS total_revenue,
        ROUND((SUM(lifetime_value) * 100.0 / (SELECT SUM(lifetime_value) FROM customer_visits))::numeric, 1) AS pct_of_revenue,
        ROUND(AVG(lifetime_value)::numeric, 2) AS avg_lifetime_value
    FROM customer_visits
    GROUP BY customer_segment
    ORDER BY num_customers DESC
''', conn)

## Quarterly Revenue Analysis

Comparing revenue across Q1-Q4 to check for seasonal patterns.

In [ ]:
pd.read_sql_query('''
    WITH quarterly_data AS (
        SELECT
            EXTRACT(YEAR FROM date_received)::text AS year,
            CASE
                WHEN EXTRACT(MONTH FROM date_received) IN (1, 2, 3) THEN 'Q1 (Jan-Mar)'
                WHEN EXTRACT(MONTH FROM date_received) IN (4, 5, 6) THEN 'Q2 (Apr-Jun)'
                WHEN EXTRACT(MONTH FROM date_received) IN (7, 8, 9) THEN 'Q3 (Jul-Sep)'
                ELSE 'Q4 (Oct-Dec)'
            END AS quarter,
            total_amount
        FROM pink_slip
    )
    SELECT
        year,
        quarter,
        COUNT(*) AS total_orders,
        ROUND(SUM(total_amount), 2) AS revenue,
        ROUND(AVG(total_amount), 2) AS avg_order_value
    FROM quarterly_data
    GROUP BY year, quarter
    ORDER BY year, quarter
''', conn)

## Revenue by Alteration Type

In [ ]:
pd.read_sql_query('''
    SELECT
        work_description AS alteration_type,
        COUNT(*) AS times_performed,
        ROUND(SUM(price)::numeric, 2) AS total_revenue,
        ROUND(AVG(price)::numeric, 2) AS avg_price,
        ROUND((SUM(price) * 100.0 / (SELECT SUM(price) FROM pink_slip_item))::numeric, 1) AS pct_of_revenue
    FROM pink_slip_item
    WHERE work_description IS NOT NULL AND work_description != ''
    GROUP BY work_description
    ORDER BY total_revenue DESC
    LIMIT 5
''', conn)